# 04 Simple RNN

Self-contained sequence-model workflow.


## 1. Imports and configuration


In [ ]:
from pathlib import Path
import sys

CURRENT_DIRECTORY = Path.cwd().resolve()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    PROJECT_ROOT = CURRENT_DIRECTORY

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


## 2. Shared training and evaluation imports


In [ ]:
import json
import time
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset

from wesad_utils.config import RANDOM_SEED, SEQUENCE_CHANNELS, WINDOW_SAMPLES
from wesad_utils.helpers import count_parameters, set_seed
from wesad_utils.training import (
    pos_weight_from_labels,
    save_model_artifacts,
    train_with_early_stopping,
)
from wesad_utils.evaluation import (
    binary_metrics,
    collect_probabilities,
    per_subject_metrics,
    prediction_table,
    select_threshold,
)

set_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
# Shared setup is imported above; model architecture remains defined in this notebook.


## 3. SimpleRNNClassifier model definition


In [ ]:
from __future__ import annotations

import torch
from torch import nn


class SimpleRNNClassifier(nn.Module):
    def __init__(self, input_size: int = 6, hidden_size: int = 32, dropout: float = 0.3) -> None:
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, hidden = self.rnn(x)
        final_hidden = hidden[-1]
        return self.classifier(final_hidden)


## 4. Load sequence tensors


In [ ]:
sequence_dir = PROJECT_ROOT / "data" / "processed" / "sequence"
metadata_dir = PROJECT_ROOT / "data" / "processed" / "metadata"
preprocessing_dir = PROJECT_ROOT / "artifacts" / "preprocessing"

X_train = torch.load(sequence_dir / "X_train.pt").float()
y_train = torch.load(sequence_dir / "y_train.pt").float()
X_validation = torch.load(sequence_dir / "X_validation.pt").float()
y_validation = torch.load(sequence_dir / "y_validation.pt").float()
X_test = torch.load(sequence_dir / "X_test.pt").float()
y_test = torch.load(sequence_dir / "y_test.pt").float()

metadata_train = pd.read_csv(metadata_dir / "windows_train.csv")
metadata_validation = pd.read_csv(metadata_dir / "windows_validation.csv")
metadata_test = pd.read_csv(metadata_dir / "windows_test.csv")
with (preprocessing_dir / "sequence_channels.json").open("r", encoding="utf-8") as handle:
    saved_sequence_channels = json.load(handle)

assert saved_sequence_channels == SEQUENCE_CHANNELS
assert X_train.shape[1:] == (WINDOW_SAMPLES, len(SEQUENCE_CHANNELS))
assert X_validation.shape[1:] == (WINDOW_SAMPLES, len(SEQUENCE_CHANNELS))
assert X_test.shape[1:] == (WINDOW_SAMPLES, len(SEQUENCE_CHANNELS))
assert torch.isfinite(X_train).all()
assert torch.isfinite(X_validation).all()
assert torch.isfinite(X_test).all()
assert set(y_train.numpy().astype(int)).issubset({0, 1})
assert set(y_validation.numpy().astype(int)).issubset({0, 1})
assert set(y_test.numpy().astype(int)).issubset({0, 1})
assert len(metadata_train) == len(y_train)
assert len(metadata_validation) == len(y_validation)
assert len(metadata_test) == len(y_test)
assert np.array_equal(metadata_train["binary_label"].to_numpy(), y_train.numpy().astype(int))
assert np.array_equal(metadata_validation["binary_label"].to_numpy(), y_validation.numpy().astype(int))
assert np.array_equal(metadata_test["binary_label"].to_numpy(), y_test.numpy().astype(int))
assert set(metadata_train["subject_id"]).isdisjoint(set(metadata_validation["subject_id"]))
assert set(metadata_train["subject_id"]).isdisjoint(set(metadata_test["subject_id"]))
assert set(metadata_validation["subject_id"]).isdisjoint(set(metadata_test["subject_id"]))

X_train.shape, X_validation.shape, X_test.shape


## 5. Load window metadata


In [ ]:
metadata_validation.head(), metadata_test.head()


## 6. Create TensorDataset objects


In [ ]:
train_dataset = TensorDataset(X_train, y_train)
validation_dataset = TensorDataset(X_validation, y_validation)
test_dataset = TensorDataset(X_test, y_test)


## 7. Create DataLoaders


In [ ]:
train_dataset = TensorDataset(X_train, y_train)
validation_dataset = TensorDataset(X_validation, y_validation)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


## 8. Inspect input and output shapes


In [ ]:
model = SimpleRNNClassifier(input_size=X_train.shape[2]).to(device)
print(X_train[:4].shape)
print(model(X_train[:4].to(device)).shape)


## 9. Define loss and optimizer


In [ ]:
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


## 10. Train without imbalance correction


In [ ]:
set_seed(42)
unweighted_model = SimpleRNNClassifier(input_size=X_train.shape[2]).to(device)
unweighted_criterion = torch.nn.BCEWithLogitsLoss()
unweighted_optimizer = torch.optim.Adam(unweighted_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
unweighted_model, unweighted_history, unweighted_training_summary = train_with_early_stopping(unweighted_model, train_loader, validation_loader, unweighted_criterion, unweighted_optimizer, device, MAX_EPOCHS, PATIENCE, gradient_clip=1.0)
unweighted_history.tail()


## 11. Train with class weighting


In [ ]:
set_seed(42)
model = SimpleRNNClassifier(input_size=X_train.shape[2]).to(device)
pos_weight = pos_weight_from_labels(y_train, device)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

model, history, weighted_training_summary = train_with_early_stopping(
    model,
    train_loader,
    validation_loader,
    criterion,
    optimizer,
    device,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    gradient_clip=1.0,
)
history.tail()


## 12. Use early stopping


In [ ]:
unweighted_validation_probabilities, unweighted_validation_true = collect_probabilities(unweighted_model, validation_loader, device)
unweighted_threshold, _ = select_threshold(unweighted_validation_true, unweighted_validation_probabilities)
unweighted_validation_metrics = binary_metrics(
    unweighted_validation_true,
    unweighted_validation_probabilities,
    unweighted_threshold,
)

weighted_validation_probabilities, weighted_validation_true = collect_probabilities(model, validation_loader, device)
weighted_threshold, _ = select_threshold(weighted_validation_true, weighted_validation_probabilities)
weighted_validation_metrics = binary_metrics(
    weighted_validation_true,
    weighted_validation_probabilities,
    weighted_threshold,
)

unweighted_best_epoch = int(unweighted_history.loc[unweighted_history["validation_loss"].idxmin(), "epoch"])
weighted_best_epoch = int(history.loc[history["validation_loss"].idxmin(), "epoch"])

validation_variant_comparison = pd.DataFrame(
    [
        {
            "method": "no_correction",
            "best_epoch": unweighted_best_epoch,
            "threshold": unweighted_threshold,
            "macro_f1": unweighted_validation_metrics["macro_f1"],
            "stress_recall": unweighted_validation_metrics["stress_recall"],
            "average_precision": unweighted_validation_metrics["average_precision"],
        },
        {
            "method": "class_weight",
            "best_epoch": weighted_best_epoch,
            "threshold": weighted_threshold,
            "macro_f1": weighted_validation_metrics["macro_f1"],
            "stress_recall": weighted_validation_metrics["stress_recall"],
            "average_precision": weighted_validation_metrics["average_precision"],
        },
    ]
)
display(validation_variant_comparison)

if weighted_validation_metrics["macro_f1"] >= unweighted_validation_metrics["macro_f1"]:
    selected_model = model
    selected_history = history
    selected_threshold = weighted_threshold
    selected_validation_probabilities = weighted_validation_probabilities
    selected_validation_true = weighted_validation_true
    selected_validation_metrics = weighted_validation_metrics
    selected_imbalance_method = "class_weight"
    selected_best_epoch = weighted_best_epoch
    selected_training_summary = weighted_training_summary
else:
    selected_model = unweighted_model
    selected_history = unweighted_history
    selected_threshold = unweighted_threshold
    selected_validation_probabilities = unweighted_validation_probabilities
    selected_validation_true = unweighted_validation_true
    selected_validation_metrics = unweighted_validation_metrics
    selected_imbalance_method = "no_correction"
    selected_best_epoch = unweighted_best_epoch
    selected_training_summary = unweighted_training_summary

model = selected_model
history = selected_history
threshold = selected_threshold
validation_probabilities = selected_validation_probabilities
validation_true = selected_validation_true
validation_metrics = {
    **selected_validation_metrics,
    "selected_imbalance_method": selected_imbalance_method,
    "best_validation_epoch": selected_best_epoch,
    "variant_comparison": validation_variant_comparison.to_dict(orient="records"),
}

threshold, selected_imbalance_method, validation_metrics


## 13. Select validation threshold


In [ ]:
# The previous cell selected the imbalance method and threshold using validation macro-F1.
threshold, selected_imbalance_method, validation_metrics


## 14. Evaluate test data


In [ ]:
inference_start_time = time.perf_counter()
test_probabilities, test_true = collect_probabilities(model, test_loader, device)
inference_time_seconds = time.perf_counter() - inference_start_time
test_metrics = binary_metrics(test_true, test_probabilities, threshold)
test_metrics = {**test_metrics, "inference_time_seconds": inference_time_seconds}
test_metrics


## 15. Evaluate each test subject


In [ ]:
subject_metrics = per_subject_metrics(metadata_test, test_true, test_probabilities, threshold)
subject_metrics


## 16. Plot training curves and confusion matrix


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
history.plot(x="epoch", y=["train_loss", "validation_loss"], ax=ax)
ax.set_title("Training history")
ax.set_ylabel("BCE loss")
plt.show()

cm = np.array(test_metrics["confusion_matrix"])
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], labels=["non-stress", "stress"])
ax.set_yticks([0, 1], labels=["non-stress", "stress"])
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center")
fig.colorbar(im, ax=ax)
plt.show()


## 17. Plot ROC and precision-recall curves


In [ ]:
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(test_true, test_probabilities, ax=axes[0])
PrecisionRecallDisplay.from_predictions(test_true, test_probabilities, ax=axes[1])
plt.tight_layout()
plt.show()


## 18. Save model and results


In [ ]:
artifact_dir = PROJECT_ROOT / "artifacts" / "models" / "rnn"
test_predictions = prediction_table(metadata_test, test_true, test_probabilities, threshold)

save_model_artifacts(
    artifact_dir=artifact_dir,
    model=model,
    model_config={ "model": "SimpleRNNClassifier", "input_size": int(X_train.shape[2]), "hidden_size": 32, "dropout": 0.3,
        "parameter_count": count_parameters(model),
        "selected_imbalance_method": selected_imbalance_method,
        "best_validation_epoch": selected_best_epoch,
        "random_seed": RANDOM_SEED, "uses_class_weighting": selected_imbalance_method == "class_weight" },
    threshold=threshold,
    history=history,
    validation_metrics=validation_metrics,
    test_metrics=test_metrics,
    per_subject=subject_metrics,
    test_predictions=test_predictions,
    training_summary={**selected_training_summary, "selected_imbalance_method": selected_imbalance_method, "inference_time_seconds": inference_time_seconds},
)
artifact_dir
